# Fine-Tuning BERT: From Fundamentals to Multi-Task Implementations

This notebook provides an in-depth, production-ready guide to fine-tuning **BERT (Bidirectional Encoder Representations from Transformers)**.

### Notebook Structure:
1. **Part 1: Fine-Tuning with Hugging Face `Trainer`** (Single Sentence Classification on IMDB sentiment analysis)
2. **Part 2: Model Inspection & Evaluation** (Inspecting transformer architecture, metrics, inference pipelines)
3. **Part 3: Model Publishing to Hugging Face Hub** (Authentication, token scopes, and namespace management)
4. **Part 4: Deep Dive into Hyperparameters & Custom PyTorch Training Loop** (Warmup math, scheduler, AdamW, gradient clipping)
5. **Part 5: Fine-Tuning BERT for Different NLP Tasks**:
   - **Task 1**: Text Classification (Custom PyTorch Loop)
   - **Task 2**: Token Classification / Named Entity Recognition (NER) & Subword Alignment
   - **Task 3**: Extractive Question Answering (`BertForQuestionAnswering` on Context + Question)
   - **Task 4**: Sentence Pair Classification / Natural Language Inference (`token_type_ids`)
6. **Part 6: Hyperparameter Cheat Sheet & Best Practices**


In [1]:
# =============================================================================
# 1. HARDWARE & RUNTIME CONFIGURATION
# =============================================================================
import os

# Force execution on GPU 0.
# In multi-GPU nodes (e.g. 8x H100), running single-process notebooks with multi-GPU can cause
# torch.nn.DataParallel NCCL broadcast collisions. Selecting a single GPU provides dedicated 80GB VRAM
# and avoids NCCL Error 2.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Configure NCCL communication settings for robust logging and fallback protocols
os.environ["NCCL_DEBUG"] = "INFO"         # Print detailed NCCL logs if communication issues arise
os.environ["NCCL_P2P_DISABLE"] = "1"     # Disable direct Peer-to-Peer NVLink access if restricted in container
os.environ["NCCL_IB_DISABLE"] = "1"      # Disable InfiniBand fallback to socket communication

import torch
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

# Verify the active hardware device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch Version:", torch.__version__)
print("Device configured:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


/nuvodata/User_data/ak57139k/anmolpro/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch Version: 2.8.0+cu128
Device configured: NVIDIA H100 80GB HBM3


/nuvodata/User_data/ak57139k/anmolpro/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:829: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


### Loading the IMDB Sentiment Dataset
The IMDB dataset consists of **50,000 movie reviews** labeled as binary sentiment:
- `0`: Negative review
- `1`: Positive review

Dataset split: 25,000 train, 25,000 test, and 50,000 unsupervised.


In [2]:
# Download and load the standard IMDB dataset from the Hugging Face Hub
dataset = load_dataset("stanfordnlp/imdb")
dataset


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [13]:
# For fast experimentation and educational walkthrough, select a subset:
# - 1,000 examples for training (125 steps at batch size 8)
# - 500 examples for evaluation / testing
# (In production, use the entire dataset of 25,000 rows by using dataset['train'] directly)
train_dataset = dataset['train']
test_dataset = dataset['test']

print(f"Selected Train subset: {len(train_dataset)} samples")
print(f"Selected Test subset:  {len(test_dataset)} samples")


Selected Train subset: 25000 samples
Selected Test subset:  25000 samples


### Initializing the Tokenizer
BERT uses **WordPiece tokenization** with a 30,522 token vocabulary (`bert-base-uncased`):
- Lowercases all input text (`uncased`).
- Adds special tokens: `[CLS]` (token ID 101) at the start, and `[SEP]` (token ID 102) at the end.
- Splits out-of-vocabulary words into subwords prefixed with `##` (e.g. `"unbelievable"` -> `["un", "##belie", "##vable"]`).


In [14]:
# Load the pretrained tokenizer matching the bert-base-uncased architecture
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Inspect dataset features: text (string) and label (ClassLabel: 'neg'=0, 'pos'=1)
print("Dataset Features:", dataset['train'].features)


Dataset Features: {'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}


In [15]:
# =============================================================================
# TOKENIZATION & PREPROCESSING PIPELINE
# =============================================================================
def tokenize_fn(examples):
    """
    Tokenizes raw text batches:
    - padding='max_length': Pads shorter sequences with [PAD] (token ID 0) up to max_length.
    - truncation=True: Cuts off sequences exceeding max_length to fit within GPU memory.
    - max_length=256: Balances context coverage against quadratic O(N^2) self-attention compute.
    """
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

def preprocess(ds):
    """
    Applies tokenization, column renaming, and PyTorch tensor formatting in a streamlined pipeline:
    1. map(): Runs tokenization in batches across dataset rows and removes raw text column to save RAM.
    2. rename_column(): Renames 'label' to 'labels' as required by Hugging Face Trainer & PyTorch loss.
    3. set_format(): Converts columns to torch.Tensor for PyTorch DataLoader ingestion.
    """
    ds = ds.map(tokenize_fn, batched=True, remove_columns=["text"])
    ds = ds.rename_column("label", "labels")
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    return ds

# Execute preprocessing on train and test subsets
train_dataset = preprocess(train_dataset)
test_dataset = preprocess(test_dataset)

# Inspect the format of a processed sample tensor
sample = train_dataset[0]
print("Sample keys:", sample.keys())
print("input_ids shape:", sample['input_ids'].shape)
print("attention_mask shape:", sample['attention_mask'].shape)
print("label:", sample['labels'])


Map: 100%|██████████| 25000/25000 [00:02<00:00, 8796.39 examples/s] 

Sample keys: dict_keys(['labels', 'input_ids', 'attention_mask'])
input_ids shape: torch.Size([256])
attention_mask shape: torch.Size([256])
label: tensor(0)


### Loading Pretrained BERT for Sequence Classification

When we load `BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)`:
- The core **12-layer Transformer encoder** is loaded with weights pretrained via Masked Language Modeling (MLM) and Next Sentence Prediction (NSP).
- The old MLM/NSP heads are discarded (hence the `UNEXPECTED` report).
- A brand new, randomly initialized classification head (`Linear(in_features=768, out_features=2)`) is attached to the `[CLS]` pooled output (hence the `MISSING` report).


In [16]:
# Load model architecture with a 2-class classification head
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Count total vs trainable parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Model Parameters:     {total_params:,}")
print(f"Trainable Parameters:        {trainable_params:,}")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11545.61it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpo

Total Model Parameters:     109,483,778
Trainable Parameters:        109,483,778


### Detailed Breakdown of Hyperparameters in `TrainingArguments`

Fine-tuning BERT requires careful hyperparameter tuning. Unlike training from scratch, pretrained weights are already near an optimal manifold. Here is what every parameter controls:

| Hyperparameter | Value | Purpose & Rationale |
| :--- | :---: | :--- |
| **`output_dir`** | `./bert_finetuned_imdb` | Directory where trained checkpoints, configuration, and artifacts will be saved. |
| **`num_train_epochs`** | `2` | Number of complete passes over the dataset. BERT typically converges in 2-4 epochs. |
| **`per_device_train_batch_size`** | `8` | Micro-batch size per GPU. Standard fine-tuning batch sizes range from 8 to 32. |
| **`per_device_eval_batch_size`** | `8` | Batch size during validation. Can be larger than train batch size since gradients are not stored. |
| **`learning_rate`** | `2e-5` | **Crucial:** Small learning rate ($2 \times 10^{-5}$). Large LR ($>10^{-3}$) destroys pretrained representations (*catastrophic forgetting*). |
| **`weight_decay`** | `0.01` | L2 weight regularization applied to weight matrices to prevent overfitting on small datasets. |
| **`logging_strategy`** & **`logging_steps`** | `"steps"`, `10` | Prints training loss every 10 gradient steps. |
| **`eval_strategy`** & **`eval_steps`** | `"steps"`, `25` | Evaluates validation loss and accuracy every 25 steps to detect overfitting early. |
| **`save_strategy`** & **`save_steps`** | `"steps"`, `50` | Saves model checkpoints every 50 steps. |
| **`save_total_limit`** | `2` | Keeps only the top 2 best/most recent checkpoints on disk, deleting older ones to save storage. |
| **`load_best_model_at_end`** | `True` | Automatically loads the best model checkpoint when training finishes based on `metric_for_best_model`. |
| **`metric_for_best_model`** | `"accuracy"` | Optimization target metric for selecting the best checkpoint. |
| **`report_to`** | `"none"` | Disables external cloud metric logging (e.g. WandB, Comet) for clean local execution. |


In [17]:
import numpy as np
from transformers import Trainer, TrainingArguments

# Evaluation metric computation callback
def compute_metrics(eval_pred):
    """
    Computes classification accuracy from model logits:
    - eval_pred: Tuple of (logits, true_labels)
    - np.argmax(logits, axis=-1): Extracts the predicted class index (0 or 1)
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = (predictions == labels).mean()
    return {"accuracy": float(accuracy)}

# Instantiate TrainingArguments with all documented hyperparameters
training_args = TrainingArguments(
    output_dir="./bert_finetuned_imdb",       # Destination directory for model artifacts
    num_train_epochs=2,                       # Total passes over the training data
    per_device_train_batch_size=8,            # Batch size per GPU for training
    per_device_eval_batch_size=8,             # Batch size per GPU for evaluation
    learning_rate=2e-5,                       # Low learning rate for fine-tuning stability
    weight_decay=0.01,                        # L2 regularization penalty
    logging_strategy="steps",                 # Log on a per-step basis
    logging_steps=10,                         # Print metrics every 10 steps
    eval_strategy="steps",                    # Evaluate at step intervals
    eval_steps=25,                            # Run evaluation every 25 steps
    save_strategy="steps",                    # Checkpoint at step intervals
    save_steps=50,                            # Save checkpoint every 50 steps
    save_total_limit=2,                       # Keep at most 2 checkpoints
    load_best_model_at_end=True,              # Restore best model upon completion
    metric_for_best_model="accuracy",         # Determine 'best' checkpoint using accuracy
    report_to="none",                         # Local stdout logging only
)

# Initialize the Hugging Face Trainer
trainer = Trainer(
    model=model,                              # Target BERT model
    args=training_args,                       # Training configuration
    train_dataset=train_dataset,              # Processed training subset
    eval_dataset=test_dataset,                # Processed validation subset
    compute_metrics=compute_metrics,          # Evaluation callback
)


In [ ]:
# Start model fine-tuning
trainer.train()


Step,Training Loss,Validation Loss,Accuracy
25,0.688795,0.637285,0.606440
50,0.431253,0.524467,0.735560
75,0.403372,0.350960,0.852960
100,0.387961,0.296851,0.877200
125,0.325473,0.492884,0.813080
150,0.372331,0.295641,0.880720
175,0.202559,0.343257,0.879920
200,0.253719,0.286484,0.883240
225,0.232244,0.327538,0.889320
250,0.437261,0.399174,0.854480


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.76it/s]


In [ ]:
# Save the fine-tuned model weights and tokenizer to disk
trainer.save_model("./bert-finetuned-imdb")
tokenizer.save_pretrained("./bert-finetuned-imdb")
print("Model and tokenizer saved to ./bert-finetuned-imdb")

# Run full evaluation on test set
final_metrics = trainer.evaluate()
print("Final Test Evaluation Metrics:", final_metrics)


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.23it/s]

Model and tokenizer saved to ./bert-finetuned-imdb


Training Loss,Validation Loss,Step,Accuracy
0.000358,0.000280,250,1.000000


Final Test Evaluation Metrics: {'eval_loss': 0.00027966342167928815, 'eval_accuracy': 1.0}


### Local Inference & Prediction Pipeline
Using Hugging Face `pipeline` with our newly fine-tuned model for sentiment classification.

In [12]:
from transformers import pipeline

# Load our fine-tuned model and tokenizer into an inference pipeline
inference_tokenizer = BertTokenizer.from_pretrained('./bert-finetuned-imdb')
inference_model = BertForSequenceClassification.from_pretrained('./bert-finetuned-imdb')

classifier = pipeline("text-classification", model=inference_model, tokenizer=inference_tokenizer)

# Test with sample positive and negative reviews
samples = [
    "This movie was fantastic! The acting was superb and the plot was gripping.",
    "Terrible movie. The story was boring, the audio was awful, and I left the theater early."
]

for text in samples:
    res = classifier(text)[0]
    sentiment = "POSITIVE" if res['label'] == 'LABEL_1' else "NEGATIVE"
    print(f"Review: '{text}'")
    print(f"-> Prediction: {sentiment} (Confidence: {res['score']:.4f})\n")


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9792.26it/s]

Review: 'This movie was fantastic! The acting was superb and the plot was gripping.'
-> Prediction: NEGATIVE (Confidence: 0.9995)

Review: 'Terrible movie. The story was boring, the audio was awful, and I left the theater early.'
-> Prediction: NEGATIVE (Confidence: 0.9996)



### Publishing to Hugging Face Hub
To publish to Hugging Face:
1. Read token from `.env` (`HUGGINGFACE_FULL_ACCESS_TOKEN` or `HUGGINGFACE_WRITE_TOKEN`).
2. Ensure the repo name is prefixed with your username (e.g. `Akhand108/my-bert-imdb2`), because models cannot be created in arbitrary namespaces without an organization.

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from huggingface_hub import login, HfApi, create_repo

# Load secrets from .env
load_dotenv(find_dotenv(usecwd=True))
load_dotenv(".env")
load_dotenv("SLM_Experiment/.env")

token = os.getenv("HUGGINGFACE_FULL_ACCESS_TOKEN") or os.getenv("HUGGINGFACE_WRITE_TOKEN")
if not token:
    print("Note: No Hugging Face token detected in .env; skipping push to hub.")
else:
    login(token=token, add_to_git_credential=True)
    api = HfApi()
    username = api.whoami()["name"]
    print(f"Authenticated as: {username}")
    
    # Define repo under your account namespace
    repo_name = "my-bert-imdb2"
    repo_id = f"{username}/{repo_name}"
    print(f"Target Repository: https://huggingface.co/{repo_id}")
    
    # Create repo if not already created
    create_repo(repo_id=repo_id, exist_ok=True, token=token)
    
    # Push tokenizer and model weights
    # tokenizer.push_to_hub(repo_id, token=token)
    # model.push_to_hub(repo_id, token=token)
    print(f"Repository ready at: https://huggingface.co/{repo_id}")


# Fine-Tuning BERT for Different NLP Tasks

BERT's bidirectionality and self-attention mechanism allow it to excel across diverse NLP tasks by simply modifying the **output classification head** and **input formatting**.

Below, we implement and thoroughly explain fine-tuning for **4 fundamental NLP tasks**:
1. **Task 1: Text Classification with Native PyTorch Loop** (Custom Dataset, AdamW, Linear Warmup Scheduler, Gradient Clipping)
2. **Task 2: Token Classification / Named Entity Recognition (NER)** (Token-level classification with Subword WordPiece alignment)
3. **Task 3: Extractive Question Answering (QA)** (Predicting answer `start_logits` and `end_logits` in context spans)
4. **Task 4: Sentence Pair Classification / Natural Language Inference (NLI)** (Premise vs. Hypothesis relation with `token_type_ids`)


## Task 1: Text Classification (Native PyTorch Training Loop)

### Understanding the Optimization Mathematics:
- **Total Training Steps**: $\text{total\_steps} = \left(\frac{N}{\text{batch\_size}}\right) \times \text{epochs}$
  - E.g., for $N=800$ samples, $\text{batch\_size}=8$, and $\text{epochs}=3$:
  - Batches per epoch = $800 / 8 = 100$
  - Total optimizer updates = $100 \times 3 = 300$ steps.
- **Learning Rate Warmup (10%)**:
  - Initial steps ($30$ steps): Learning rate increases linearly from $0$ to peak ($2 \times 10^{-5}$).
  - Remaining steps ($270$ steps): Decays linearly from peak to $0$.
  - **Why warmup?** At step 0, the classification head weights are completely random. Large gradients from random weights would corrupt the well-tuned pretrained transformer encoder representations. Warmup allows the head to stabilize first.
- **Gradient Clipping (`max_norm=1.0`)**:
  - Rescales gradient vectors if their norm exceeds 1.0, preventing exploding gradients in deep 12-layer backpropagation.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, classification_report, f1_score
import numpy as np
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Training on:", device)

# =============================================================================
# CUSTOM PYTORCH DATASET FOR TEXT CLASSIFICATION
# =============================================================================
class TextClassificationDataset(Dataset):
    """
    Standard PyTorch Dataset for on-the-fly tokenization of raw text strings.
    Useful when reading directly from CSVs, dataframes, or raw lists without Hugging Face map().
    """
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# =============================================================================
# BERT TEXT CLASSIFIER CONTROLLER (Native PyTorch)
# =============================================================================
class BERTTextClassifier:
    """
    Encapsulates BERT Sequence Classification with a custom PyTorch training loop,
    AdamW optimizer, warmup learning rate scheduler, and gradient clipping.
    """
    def __init__(self, model_name='bert-base-uncased', num_classes=2, max_length=256):
        self.model_name = model_name
        self.num_classes = num_classes
        self.max_length = max_length
        self.tokenizer = BertTokenizerFast.from_pretrained(model_name)
        self.model = BertForSequenceClassification.from_pretrained(model_name, num_labels=num_classes)
        self.model.to(device)

    def train(self, train_texts, train_labels, epochs=2, batch_size=8, learning_rate=2e-5, weight_decay=0.01):
        """
        Executes native PyTorch training loop with full progress monitoring.
        """
        train_dataset = TextClassificationDataset(train_texts, train_labels, self.tokenizer, self.max_length)
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

        # AdamW optimizer with weight decay on non-bias/non-LayerNorm parameters
        no_decay = ['bias', 'LayerNorm.weight']
        optimizer_grouped_parameters = [
            {
                'params': [p for n, p in self.model.named_parameters() if not any(nd in n for nd in no_decay)],
                'weight_decay': weight_decay
            },
            {
                'params': [p for n, p in self.model.named_parameters() if any(nd in n for nd in no_decay)],
                'weight_decay': 0.0
            }
        ]
        optimizer = AdamW(optimizer_grouped_parameters, lr=learning_rate)

        # Linear learning rate schedule with 10% warmup steps
        total_steps = len(train_loader) * epochs
        warmup_steps = int(total_steps * 0.1)
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps
        )

        print(f"\n[Training] Total Steps: {total_steps} | Warmup Steps: {warmup_steps}")
        self.model.train()

        for epoch in range(epochs):
            total_loss = 0.0
            progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

            for batch in progress_bar:
                optimizer.zero_grad()
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                outputs = self.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )

                loss = outputs.loss
                total_loss += loss.item()

                loss.backward()

                # Gradient clipping to prevent gradient explosion
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)

                optimizer.step()
                scheduler.step()

                progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

            avg_loss = total_loss / len(train_loader)
            print(f"Epoch {epoch+1} Completed | Average Loss: {avg_loss:.4f}")

    def evaluate(self, test_texts, test_labels, batch_size=8):
        """
        Evaluates accuracy, F1 score, and prints a comprehensive classification report.
        """
        test_dataset = TextClassificationDataset(test_texts, test_labels, self.tokenizer, self.max_length)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
        self.model.eval()

        predictions = []
        true_labels = []

        with torch.no_grad():
            for batch in tqdm(test_loader, desc='Evaluating'):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                preds = torch.argmax(logits, dim=-1)

                predictions.extend(preds.cpu().numpy())
                true_labels.extend(labels.cpu().numpy())

        acc = accuracy_score(true_labels, predictions)
        f1 = f1_score(true_labels, predictions, average='weighted')
        print(f"\nEvaluation Accuracy: {acc:.4f} | Weighted F1 Score: {f1:.4f}")
        print("\nClassification Report:\n", classification_report(true_labels, predictions))
        return acc, f1


In [ ]:
# Demonstration of Custom PyTorch Text Classification
demo_train_texts = [
    "The film was an absolute masterpiece with stellar performances.",
    "Unwatchable garbage. Horrible writing and dreadful acting.",
    "An incredible experience, highly emotional and thought-provoking.",
    "Complete waste of time, completely predictable and dull.",
] * 20  # 80 training samples
demo_train_labels = [1, 0, 1, 0] * 20

demo_test_texts = [
    "A truly brilliant piece of cinema.",
    "I hated every single minute of this boring movie."
] * 10
demo_test_labels = [1, 0] * 10

classifier_pytorch = BERTTextClassifier(num_classes=2, max_length=64)
classifier_pytorch.train(demo_train_texts, demo_train_labels, epochs=1, batch_size=8, learning_rate=2e-5)
classifier_pytorch.evaluate(demo_test_texts, demo_test_labels, batch_size=8)


## Task 2: Token Classification / Named Entity Recognition (NER)

### How BERT Handles Token Classification:
- Unlike text classification where only the `[CLS]` token is pooled, in Token Classification, **every token produces a prediction** via `BertForTokenClassification`.
- Output shape: `(batch_size, sequence_length, num_labels)`.

### The Core Challenge: Subword WordPiece Alignment!
Consider the word **`"HuggingFace"`** with entity label `B-ORG`:
- Pre-tokenized word: `"HuggingFace"` $\rightarrow$ label: `B-ORG` (ID 1)
- BERT Tokenizer output: `['Hu', '##gging', 'Face']` (3 subwords!)
- Special tokens: `[CLS]` and `[SEP]` do not correspond to any word.

**The Solution:**
- We label the **first subword** (`'Hu'`) with the entity label (`B-ORG`).
- We label subsequent subwords (`'##gging'`, `'Face'`) and special tokens with **`-100`**.
- PyTorch's `nn.CrossEntropyLoss(ignore_index=-100)` automatically skips tokens with label `-100` during backpropagation!


In [ ]:
from transformers import BertForTokenClassification

# NER Label Mapping (IOB2 Format: Beginning, Inside, Outside)
ner_labels = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]
label2id = {label: i for i, label in enumerate(ner_labels)}
id2label = {i: label for i, label in enumerate(ner_labels)}

print("Label Mapping:", label2id)

# Initialize BERT Token Classifier for 7 NER classes
ner_model = BertForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(ner_labels),
    id2label=id2label,
    label2id=label2id
).to(device)


In [ ]:
# =============================================================================
# SUBWORD ALIGNMENT FUNCTION FOR NER
# =============================================================================
def align_labels_with_tokens(words, word_labels, tokenizer, max_length=64):
    """
    Aligns word-level NER tags to WordPiece subword tokens.
    Special tokens and subsequent subwords receive -100 so loss ignores them.
    """
    tokenized_inputs = tokenizer(
        words,
        is_split_into_words=True,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    
    # word_ids() maps each token to its original word index (None for [CLS], [SEP], [PAD])
    word_ids = tokenized_inputs.word_ids(batch_index=0)
    aligned_labels = []
    current_word = None
    
    for word_id in word_ids:
        if word_id is None:
            aligned_labels.append(-100)  # Special token ([CLS], [SEP], [PAD])
        elif word_id != current_word:
            current_word = word_id
            aligned_labels.append(label2id[word_labels[word_id]])  # First subword
        else:
            aligned_labels.append(-100)  # Subsequent subword token (ignore)
            
    tokenized_inputs["labels"] = torch.tensor([aligned_labels], dtype=torch.long)
    return tokenized_inputs

# Demonstration
sample_words = ["Sundar", "Pichai", "leads", "Google", "in", "California"]
sample_ner   = ["B-PER",  "I-PER",  "O",     "B-ORG",  "O",  "B-LOC"]

fast_tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
aligned_batch = align_labels_with_tokens(sample_words, sample_ner, fast_tokenizer)

tokens = fast_tokenizer.convert_ids_to_tokens(aligned_batch["input_ids"][0][:10])
labels = aligned_batch["labels"][0][:10].tolist()

print("Tokens: ", tokens)
print("Labels: ", labels)

# Forward pass to compute token classification loss
with torch.no_grad():
    input_ids = aligned_batch["input_ids"].to(device)
    attention_mask = aligned_batch["attention_mask"].to(device)
    target_labels = aligned_batch["labels"].to(device)
    
    output = ner_model(input_ids=input_ids, attention_mask=attention_mask, labels=target_labels)
    print(f"NER Cross-Entropy Loss: {output.loss.item():.4f}")
    print(f"Logits shape: {output.logits.shape} (batch_size, seq_len, num_classes)")


## Task 3: Extractive Question Answering (SQuAD Style)

### How Extractive QA Works:
- In Extractive QA, the model is given a **(Question, Context)** pair.
- BERT inputs both formatted as: `[CLS] Question [SEP] Context [SEP]`.
- **Model Head (`BertForQuestionAnswering`)**:
  - Projects each token embedding into **2 scalar outputs**: `start_logits` and `end_logits`.
  - `start_logits[i]`: Probability that the answer span begins at token position `i`.
  - `end_logits[j]`: Probability that the answer span ends at token position `j`.
- The answer text is extracted by slicing tokens from index `argmax(start_logits)` to `argmax(end_logits)`.


In [ ]:
from transformers import BertForQuestionAnswering

# Load BertForQuestionAnswering
qa_model = BertForQuestionAnswering.from_pretrained("bert-base-uncased").to(device)
qa_tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

# Example Context and Question
context = "DeepMind is a research laboratory that develops general artificial intelligence. It was founded in London in 2010."
question = "Where was DeepMind founded?"

# Tokenize Question and Context together
inputs = qa_tokenizer(question, context, return_tensors="pt", max_length=128, truncation=True).to(device)

# Forward pass
with torch.no_grad():
    qa_outputs = qa_model(**inputs)

start_logits = qa_outputs.start_logits
end_logits = qa_outputs.end_logits

# Find most likely start and end token indices
start_idx = torch.argmax(start_logits, dim=-1).item()
end_idx = torch.argmax(end_logits, dim=-1).item()

print(f"Question:     {question}")
print(f"Start Logits shape: {start_logits.shape}")
print(f"End Logits shape:   {end_logits.shape}")
print(f"Predicted Token Span: [{start_idx}:{end_idx+1}]")


## Task 4: Sentence Pair Classification (NLI & Paraphrase Detection)

### How BERT Processes Sentence Pairs:
- Input Structure: `[CLS] Premise [SEP] Hypothesis [SEP]`
- **Token Type IDs (Segment Embeddings)**:
  - Tokens of Sentence A (Premise) have `token_type_id = 0`
  - Tokens of Sentence B (Hypothesis) have `token_type_id = 1`
- Cross-attention allows BERT to attend bidirectionally between Sentence A and Sentence B across all layers.
- Classification is performed using the final pooled `[CLS]` token vector.
- **Common Applications**: Natural Language Inference (Entailment/Contradiction/Neutral), Duplicate Question Detection (Quora Question Pairs), Semantic Similarity.


In [ ]:
# Demonstration of Sentence Pair Encoding
pair_tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

sentence_a = "A soccer player kicks the ball into the net."
sentence_b = "An athlete scores a goal in a football match."

pair_encoding = pair_tokenizer(
    sentence_a,
    sentence_b,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

print("Sentence A:", sentence_a)
print("Sentence B:", sentence_b)
print("\nEncoded input_ids:", pair_encoding["input_ids"][0][:16])
print("token_type_ids:    ", pair_encoding["token_type_ids"][0][:16])
print("\nNotice: token_type_ids are 0 for Sentence A and 1 for Sentence B!")

# Instantiate Sentence Pair Classifier (3 classes: Entailment=0, Neutral=1, Contradiction=2)
nli_model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3).to(device)

with torch.no_grad():
    nli_inputs = {k: v.to(device) for k, v in pair_encoding.items()}
    nli_outputs = nli_model(**nli_inputs)
    print("NLI Logits shape:", nli_outputs.logits.shape, "-> (batch_size, 3 classes)")


## Summary Reference: BERT Multi-Task Hyperparameter Cheat Sheet

| NLP Task | Hugging Face Class | Input Format | Target / Labels | Recommended Learning Rate | Recommended Batch Size | Recommended Epochs |
| :--- | :--- | :--- | :--- | :---: | :---: | :---: |
| **Text Classification** | `BertForSequenceClassification` | `[CLS] Text [SEP]` | Scalar class ID ($0$ to $C-1$) | `2e-5` to `3e-5` | `16` or `32` | `2` to `4` |
| **Token Classification (NER)** | `BertForTokenClassification` | `[CLS] Word1 Word2 ... [SEP]` | Label vector per token (subwords set to `-100`) | `3e-5` to `5e-5` | `8` or `16` | `3` to `5` |
| **Question Answering** | `BertForQuestionAnswering` | `[CLS] Question [SEP] Context [SEP]` | Start token index & End token index | `3e-5` | `8` or `16` | `2` to `3` |
| **Sentence Pair (NLI)** | `BertForSequenceClassification` | `[CLS] SentA [SEP] SentB [SEP]` | Scalar relation ID ($0$ to $C-1$) with `token_type_ids` | `2e-5` | `16` or `32` | `3` |

### Golden Rules for Fine-Tuning BERT:
1. **Never use standard pretraining learning rates ($10^{-3}$)**: Always use fine-tuning rates between $2 \times 10^{-5}$ and $5 \times 10^{-5}$.
2. **Always use Warmup**: Warm up the learning rate for the first 10% of steps to prevent the randomly initialized head from destabilizing the encoder.
3. **Clip Gradients**: Apply `clip_grad_norm_(max_norm=1.0)` to maintain numerical stability in deep Transformer layers.
4. **Truncate smartly**: Quadratic self-attention complexity $O(L^2)$ means reducing `max_length` from 512 to 256 speeds up training by $\sim 4\times$ and halves VRAM usage.
